1. Import Libraries

In [18]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [19]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from xgboost import XGBRegressor, XGBClassifier

import shap

import warnings
warnings.filterwarnings("ignore")

2. Load Dataset

In [20]:
df = pd.read_csv(
    "../data/MachineLearningRating_v3.txt",
    sep="|"
)

In [21]:
df.head()

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


3. Create Derived Features

In [22]:
# Vehicle Age
df["VehicleAge"] = 2015 - df["RegistrationYear"]

# Margin
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

# Claim Indicator
df["HasClaim"] = np.where(df["TotalClaims"] > 0, 1, 0)

4. Handle Missing Values

In [23]:
#Check missing values
missing = df.isnull().sum().sort_values(ascending=False)

missing.head(20)

NumberOfVehiclesInFleet    1000098
CrossBorder                 999400
CustomValueEstimate         779642
WrittenOff                  641901
Converted                   641901
Rebuilt                     641901
NewVehicle                  153295
Bank                        145961
AccountType                  40232
Gender                        9536
MaritalStatus                 8259
mmcode                         552
NumberOfDoors                  552
VehicleIntroDate               552
cubiccapacity                  552
Cylinders                      552
Model                          552
make                           552
kilowatts                      552
bodytype                       552
dtype: int64

In [24]:
#Drop columns with too many missing values
threshold = len(df) * 0.5

df = df.dropna(thresh=threshold, axis=1)

5. Select Features

In [25]:
#Use important features only
features = [
    "Province",
    "PostalCode",
    "VehicleType",
    "make",
    "Model",
    "RegistrationYear",
    "VehicleAge",
    "kilowatts",
    "Cylinders",
    "SumInsured",
    "CalculatedPremiumPerTerm",
    "Gender"
]

6. Claim Severity Model Dataset

In [26]:
#Predict TotalClaims ONLY where claims > 0.
claims_df = df[df["TotalClaims"] > 0].copy()

In [27]:
print(claims_df.columns.tolist())

['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims', 'VehicleAge', 'Margin', 'HasClaim']


In [28]:
#Target
X = claims_df[features]
y = claims_df["TotalClaims"]

7. Identify Categorical and Numerical Columns

In [29]:
categorical_cols = X.select_dtypes(include=["object"]).columns

numerical_cols = X.select_dtypes(exclude=["object"]).columns

8. Create Preprocessing Pipeline

In [30]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

9. Split Data

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

10. Create Models

In [32]:
#Linear Regression
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_preds = linear_model.predict(X_test)

In [33]:
#Random Forest
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

In [34]:
#XGBoost
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

11. Evaluate Models

In [35]:
#Run Evaluations
from src.modeling import evaluate_regression

In [36]:
#Linear Regression Metrics
linear_rmse, linear_r2 = evaluate_regression(y_test, linear_preds)

print("Linear Regression RMSE:", linear_rmse)
print("Linear Regression R2:", linear_r2)

Linear Regression RMSE: 36972.780934098504
Linear Regression R2: 0.15001508525027019


In [37]:
#Random Forest Metrics
rf_rmse, rf_r2 = evaluate_regression(y_test, rf_preds)

print("Random Forest RMSE:", rf_rmse)
print("Random Forest R2:", rf_r2)

Random Forest RMSE: 36373.25811001061
Random Forest R2: 0.1773570280094351


In [38]:
#XGBoost Metrics
xgb_rmse, xgb_r2 = evaluate_regression(y_test, xgb_preds)

print("XGBoost RMSE:", xgb_rmse)
print("XGBoost R2:", xgb_r2)

XGBoost RMSE: 36008.0174063123
XGBoost R2: 0.19379515706764783


12. Model Comparison Table

In [39]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],
    
    "RMSE": [
        linear_rmse,
        rf_rmse,
        xgb_rmse
    ],
    
    "R2 Score": [
        linear_r2,
        rf_r2,
        xgb_r2
    ]
})

comparison

,Model,RMSE,R2 Score
0,Linear Regression,36972.780934,0.150015
1,Random Forest,36373.258110,0.177357
2,XGBoost,36008.017406,0.193795


# Model Evaluation Interpretation

Three regression models were developed and evaluated to predict insurance claim severity (`TotalClaims`):

- Linear Regression
- Random Forest Regressor
- XGBoost Regressor

The models were evaluated using:
- RMSE (Root Mean Squared Error)
- R² Score (Coefficient of Determination)

## Interpretation of Results

### 1. XGBoost Performed Best
XGBoost achieved the:
- Lowest RMSE (36,008)
- Highest R² Score (0.194)

This indicates that XGBoost produced the most accurate predictions among the three models and explained approximately 19.4% of the variance in insurance claim amounts.

The superior performance of XGBoost suggests that:
- Insurance claim behavior is nonlinear
- Complex interactions exist between policy, customer, and vehicle variables
- Tree-based ensemble models capture risk patterns better than simple linear relationships

---

### 2. Random Forest Performed Better Than Linear Regression
Random Forest achieved:
- Lower RMSE than Linear Regression
- Higher R² score

This indicates that ensemble-based learning methods improve prediction accuracy by capturing nonlinear relationships and interactions among variables.

However, the performance gain compared to XGBoost was relatively small.

---

### 3. Linear Regression Had the Weakest Performance
Linear Regression produced:
- Highest RMSE
- Lowest R² Score

This suggests that the relationship between predictor variables and claim amounts is not purely linear.

Insurance claim data is often:
- highly skewed
- influenced by rare high-cost claims
- affected by interactions between risk variables

Linear Regression struggles to model these complex behaviors effectively.

---

# Business Interpretation

The results demonstrate that machine learning approaches, particularly XGBoost, can improve ACIS's ability to estimate future claim severity and support risk-based pricing decisions.

Although the R² scores are modest, this is common in real-world insurance datasets because claim amounts are highly volatile and influenced by many external factors.

The findings suggest that:
- geographic factors,
- vehicle characteristics,
- customer demographics,
- and premium-related variables

all contribute to claim behavior, but additional features may further improve predictive performance.

---

# Recommendation

Based on model performance:
- XGBoost is recommended as the preferred severity prediction model for ACIS.
- Random Forest may serve as a secondary benchmark model.
- Linear Regression can be retained as a simple baseline for comparison.

Future improvements could include:
- additional feature engineering,
- hyperparameter tuning,
- handling extreme outliers,
- balancing claim distributions,
- and incorporating external risk variables such as crime rates or weather conditions.